In [ ]:
import numpy as np
import re
from database import db
from scipy.spatial import distance
from pymongo import MongoClient
from urllib.parse import quote_plus

# 1. DB 연결
user = "JS"
password = quote_plus("Z26SdTRgqMadKJST") 
cluster_url = "cluster0.qpamvvh.mongodb.net"
uri = f"mongodb+srv://{user}:{password}@{cluster_url}/sesac_final?retryWrites=true&w=majority"

client = MongoClient(uri)
db = client['sesac_final']

pipeline = [
    # address에서 동 이름 추출 (예: "영등포구 영등포동2가 440" → "영등포동2가")
    {
        "$addFields": {
            "dong_name": {
                "$regexFind": {
                    "input": "$address",
                    "regex": r"(\S+동\S*)\s"  # "XX동" 패턴 추출
                }
            }
        }
    },
    {
        "$addFields": {
            "dong_name": "$dong_name.match"
        }
    },
    {"$match": {"dong_name": {"$ne": None}}},
    {
        "$group": {
            "_id": "$dong_name",
            "avg_traffic":     {"$avg": "$category_scores.traffic"},
            "avg_convenience": {"$avg": "$category_scores.convenience"},
            "avg_green":       {"$avg": "$category_scores.green"},
            "avg_play":        {"$avg": "$category_scores.play"},
            "avg_health":      {"$avg": "$category_scores.health"},
            "avg_living":      {"$avg": "$category_scores.living"},
            "avg_safety":      {"$avg": "$category_scores.safety"},
            "house_count":     {"$sum": 1}
        }
    },
    {"$match": {"house_count": {"$gte": 2}}},  # 매물 2개 이상인 동만
    {"$out": "dong_profiles"}  # 결과를 dong_profiles 컬렉션에 저장
]

db.houses.aggregate(pipeline)
print("완료! dong_profiles 컬렉션 생성됨")
```

## 실행하면 이런 데이터가 생김
```
{
  "_id": "영등포동2가",
  "avg_traffic": 0.54,
  "avg_green": 0.82,
  "avg_play": 0.71,
  ...
  "house_count": 15
}

In [18]:
import numpy as np
import re
from scipy.spatial import distance
from pymongo import MongoClient
from urllib.parse import quote_plus

# 1. DB 연결
user = "JS"
password = quote_plus("Z26SdTRgqMadKJST") 
cluster_url = "cluster0.qpamvvh.mongodb.net"
uri = f"mongodb+srv://{user}:{password}@{cluster_url}/sesac_final?retryWrites=true&w=majority"

client = MongoClient(uri)
db = client['sesac_final']

pipeline = [
    {
        "$addFields": {
            "dong_name": {
                "$arrayElemAt": [{"$split": ["$address", " "]}, 1]
            }
        }
    },
    {
        "$group": {
            "_id": "$dong_name",
            "avg_traffic":     {"$avg": "$category_scores.traffic"},
            "avg_convenience": {"$avg": "$category_scores.convenience"},
            "avg_green":       {"$avg": "$category_scores.green"},
            "avg_play":        {"$avg": "$category_scores.play"},
            "avg_health":      {"$avg": "$category_scores.health"},
            "avg_living":      {"$avg": "$category_scores.living"},
            "avg_safety":      {"$avg": "$category_scores.safety"},
            "house_count":     {"$sum": 1}
        }
    },
    {"$match": {"house_count": {"$gte": 2}}},
    {"$sort": {"house_count": -1}}  # $out 대신 정렬로 결과 확인
]

results = list(db.properties_test2.aggregate(pipeline))

print(f"총 {len(results)}개 동 추출됨\n")
for dong in results[:30]:
    print(f"[{dong['_id']}] 매물 {dong['house_count']}개")
    print(f"  교통: {dong['avg_traffic'] or 0:.3f} | 편의: {dong['avg_convenience'] or 0:.3f} | 녹지: {dong['avg_green'] or 0:.3f}")
    print(f"  놀이: {dong['avg_play'] or 0:.3f} | 건강: {dong['avg_health'] or 0:.3f} | 생활: {dong['avg_living'] or 0:.3f} | 안전: {dong['avg_safety'] or 0:.3f}")
    print()

총 301개 동 추출됨

[역삼동] 매물 1155개
  교통: 0.458 | 편의: 0.927 | 녹지: 0.399
  놀이: 0.934 | 건강: 0.934 | 생활: 0.987 | 안전: 0.502

[봉천동] 매물 791개
  교통: 0.539 | 편의: 0.622 | 녹지: 0.418
  놀이: 0.319 | 건강: 0.636 | 생활: 0.397 | 안전: 0.311

[신림동] 매물 744개
  교통: 0.588 | 편의: 0.703 | 녹지: 0.545
  놀이: 0.278 | 건강: 0.521 | 생활: 0.346 | 안전: 0.276

[논현동] 매물 660개
  교통: 0.360 | 편의: 0.868 | 녹지: 0.338
  놀이: 0.937 | 건강: 0.937 | 생활: 0.967 | 안전: 0.575

[화곡동] 매물 489개
  교통: 0.517 | 편의: 0.629 | 녹지: 0.633
  놀이: 0.118 | 건강: 0.634 | 생활: 0.395 | 안전: 0.681

[서초동] 매물 455개
  교통: 0.627 | 편의: 0.739 | 녹지: 0.250
  놀이: 0.923 | 건강: 0.913 | 생활: 0.944 | 안전: 0.269

[수유동] 매물 439개
  교통: 0.699 | 편의: 0.663 | 녹지: 0.601
  놀이: 0.092 | 건강: 0.602 | 생활: 0.389 | 안전: 0.727

[가산동] 매물 388개
  교통: 0.704 | 편의: 0.493 | 녹지: 0.212
  놀이: 0.561 | 건강: 0.439 | 생활: 0.700 | 안전: 0.395

[천호동] 매물 382개
  교통: 0.429 | 편의: 0.677 | 녹지: 0.577
  놀이: 0.246 | 건강: 0.817 | 생활: 0.482 | 안전: 0.386

[화양동] 매물 382개
  교통: 0.403 | 편의: 0.648 | 녹지: 0.465
  놀이: 0.699 | 건강: 0.426 | 생활: 0.606 | 안전: 0.

In [8]:
sample = db.properties_test2.find_one()
print(sample)

{'_id': 47972463, 'rent_type': '전세', 'address': '영등포구 영등포동2가 440', 'deposit': 0, 'price': 30000, 'size_m2': 38.29, 'floor': '중/9', 'options': ['에어컨', '냉장고', '세탁기', '인덕션', '신발장', '싱크대'], 'hasParking': '주차 가능', 'images': ['https://ic.zigbang.com/ic/items/47972463/1.jpg', 'https://ic.zigbang.com/ic/items/47972463/2.jpg', 'https://ic.zigbang.com/ic/items/47972463/3.jpg', 'https://ic.zigbang.com/ic/items/47972463/4.jpg', 'https://ic.zigbang.com/ic/items/47972463/5.jpg', 'https://ic.zigbang.com/ic/items/47972463/6.jpg', 'https://ic.zigbang.com/ic/items/47972463/7.jpg', 'https://ic.zigbang.com/ic/items/47972463/8.jpg', 'https://ic.zigbang.com/ic/items/47972463/9.jpg', 'https://ic.zigbang.com/ic/items/47972463/10.jpg', 'https://ic.zigbang.com/ic/items/47972463/11.jpg'], 'url': 'https://www.zigbang.com/home/oneroom/items/47972463', 'year_built': '2018-09-13', 'buildingUse': '원룸', 'location': {'type': 'Point', 'coordinates': [126.909524371813, 37.5213505664709]}, 'category_scores': {'traffic': 0

In [11]:
import re

sample_address = '영등포구 영등포동2가 440'
match = re.search(r'(\S+동\S*)', sample_address)
print(match)
print(match.group(1) if match else "None")

<re.Match object; span=(5, 11), match='영등포동2가'>
영등포동2가


In [14]:
# regex 없이 그냥 전체 주소만 출력
results_test = list(db.properties_test2.aggregate([
    {"$limit": 5},
    {"$project": {"address": 1, "category_scores": 1}}
]))

for r in results_test:
    print(r)

{'_id': 47972463, 'address': '영등포구 영등포동2가 440', 'category_scores': {'traffic': 0.5384, 'convenience': 0.5597, 'green': 0.858, 'play': 0.7362, 'health': 0.527, 'living': 0.7653, 'safety': 0.6915}}
{'_id': 47841429, 'address': '송파구 송파동 52-2', 'category_scores': {'traffic': 0.3574, 'convenience': 0.6694, 'green': 0.3849, 'play': 0.3099, 'health': 0.7064, 'living': 0.7172, 'safety': 0.4407}}
{'_id': 47972657, 'address': '동대문구 장안동 384-8', 'category_scores': {'traffic': 0.5067, 'convenience': 0.7145, 'green': 0.6067, 'play': 0.2308, 'health': 0.7352, 'living': 0.4835, 'safety': 0.8571}}
{'_id': 47972762, 'address': '강동구 성내동 148-1', 'category_scores': {'traffic': 0.4666, 'convenience': 0.7763, 'green': 0.4854, 'play': 0.4511, 'health': 0.9267, 'living': 0.5988, 'safety': 0.5019}}
{'_id': 47972800, 'address': '성동구 용답동 228-10', 'category_scores': {'traffic': 0.2886, 'convenience': 0.5185, 'green': 0.5751, 'play': 0.3922, 'health': 0.5015, 'living': 0.593, 'safety': 0.7836}}


### DB에 저장

In [19]:
import numpy as np
import re
from scipy.spatial import distance
from pymongo import MongoClient
from urllib.parse import quote_plus

# 1. DB 연결
user = "JS"
password = quote_plus("Z26SdTRgqMadKJST") 
cluster_url = "cluster0.qpamvvh.mongodb.net"
uri = f"mongodb+srv://{user}:{password}@{cluster_url}/sesac_final?retryWrites=true&w=majority"

client = MongoClient(uri)
db = client['sesac_final']

pipeline = [
    {
        "$addFields": {
            "dong_name": {
                "$arrayElemAt": [{"$split": ["$address", " "]}, 1]
            }
        }
    },
    {
        "$group": {
            "_id": "$dong_name",
            "avg_traffic":     {"$avg": "$category_scores.traffic"},
            "avg_convenience": {"$avg": "$category_scores.convenience"},
            "avg_green":       {"$avg": "$category_scores.green"},
            "avg_play":        {"$avg": "$category_scores.play"},
            "avg_health":      {"$avg": "$category_scores.health"},
            "avg_living":      {"$avg": "$category_scores.living"},
            "avg_safety":      {"$avg": "$category_scores.safety"},
            "house_count":     {"$sum": 1}
        }
    },
    {"$match": {"house_count": {"$gte": 2}}},
    {"$out": "dong_profiles"}  # db 업로드
]

db.properties_test2.aggregate(pipeline)
print("완료! dong_profiles 컬렉션 저장됨")

완료! dong_profiles 컬렉션 저장됨
